# W8D1 — Get Your Capstone Serving — Lab

**Week 8 · Day 1 · Capstone** · Lab

The last authored notebook of the bootcamp, and the only one whose second half runs on **your own
project**.

This morning showed three ways a model that works in a notebook fails to ship. You are going to
reproduce all three here, in a controlled setting, on the churn pipeline you built in W2D5 — and
each failure runs in **a genuinely separate process**, because the entire lesson evaporates if you
test in another cell. A second cell still has your fitted objects in memory. A service does not.

Then you do the same thing to your capstone, with the instructor and the TA in the room.

**Split the slot: about 60 minutes on the guided half, then 90 on your own project.** The guided
half has sanity checks. The second half has a checklist that a TA signs off, which is a harder
test.

<div dir="rtl" align="right">

# الأسبوع ٨ اليوم ١ — اجعل مشروعك يعمل خارج الدفتر

**الأسبوع الثامن · اليوم الأول · مشروع التخرّج** · معمل

آخر دفتر مؤلَّف في المعسكر، والوحيد الذي يعمل نصفه الثاني على **مشروعك أنت**.

أراك الصباح ثلاث طرق يُخفق بها نموذجٌ يعمل في الدفتر عن الوصول إلى الإنتاج. وستعيد إنتاجها الثلاث
هنا في وضع مضبوط، على مسار الانصراف الذي بنيته في الأسبوع الثاني اليوم الخامس — وكل إخفاق يعمل في
**عملية منفصلة فعلًا**، لأن الدرس كله يتبخّر إذا اختبرت في خلية أخرى. فالخلية الأخرى ما تزال تحمل
كائناتك المُدرَّبة في الذاكرة. والخدمة لا تحملها.

ثم تفعل الشيء نفسه بمشروعك، والمدرّب والمساعد في القاعة.

**اقسم الحصّة: نحو ٦٠ دقيقة للنصف المُوجَّه، ثم ٩٠ لمشروعك.** وللنصف المُوجَّه فحوص سلامة. وللنصف
الثاني قائمة يوقّعها المساعد، وهي اختبار أصعب.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Say what a model artifact has to contain, and prove it in a process that never saw your notebook.
- Reproduce the weights-only failure and read its traceback as a sentence about missing transforms.
- Reproduce the **silent** failure — refitting a transform at inference — and detect it with a test
  rather than by luck.
- Write `artifact_meta.json` with the four fields that make a prediction traceable.
- Decide what a service should do with a category it has never seen, and defend the choice.
- Run your own capstone's prediction path from a command line, in a fresh clone.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تقول ماذا يجب أن يحوي مُخرَج النموذج، وأن تُثبته في عملية لم ترَ دفترك قط.
- أن تُعيد إنتاج إخفاق «الأوزان وحدها» وأن تقرأ أثره على أنه جملة عن تحويلات مفقودة.
- أن تُعيد إنتاج الإخفاق **الصامت** — إعادة ملاءمة تحويل وقت الاستدلال — وأن تكتشفه باختبار لا
  بالحظّ.
- أن تكتب `artifact_meta.json` بالحقول الأربعة التي تجعل التنبّؤ قابلًا للتتبّع.
- أن تقرّر ماذا تفعل الخدمة بفئة لم ترَها قط، وأن تدافع عن اختيارك.
- أن تشغّل مسار التنبّؤ في مشروعك من سطر الأوامر، في نسخة جديدة من المستودع.

</div>

## About the data

**`telco_churn`, through W2D5's artefacts.** You load two things you made in week 2:
`features.parquet` (W2D4's engineered table) and `pipeline.joblib` (W2D5's fitted pipeline — a
`ColumnTransformer` with median imputation, scaling and one-hot encoding, then a logistic
regression). If you no longer have them, `load_artefact` takes the reference copies from
`shared/solutions_cache/`.

One row is one telecoms customer. The target is `churn_flag`: did they leave.

**The known problem, and today it is the subject rather than a footnote.** `pipeline.joblib` does
*not* contain everything needed to predict from raw data. Two columns — `contract_months` and
`tenure_bucket` — were engineered in W2D4, **in a notebook**, before the pipeline ever saw the data.
The artifact is therefore incomplete, and nothing about it says so. That gap is exactly the shape of
the first bug on every real serving stack, and task 1 makes you look straight at it.

**For the second half you need your own capstone data and model**, in whatever state they are in.
Bring the repo, not a screenshot.

<div dir="rtl" align="right">

## عن البيانات

**`telco_churn` عبر مُخرَجات الأسبوع الثاني اليوم الخامس.** تُحمّل شيئين صنعتهما في الأسبوع الثاني:
`features.parquet` (جدول الخصائص من اليوم الرابع) و`pipeline.joblib` (المسار المُدرَّب من اليوم
الخامس — مُحوِّل أعمدة فيه تعويض بالوسيط وتقييس وترميز أحادي، ثم انحدار لوجستي). وإن لم تعد عندك
فإن `load_artefact` يأخذ النسختين المرجعيتين من `shared/solutions_cache/`.

والصفّ الواحد عميل اتصالات. والهدف `churn_flag`: هل غادر.

**والمشكلة المعروفة، وهي اليوم الموضوع لا الهامش.** لا يحوي `pipeline.joblib` كل ما يلزم للتنبّؤ من
بيانات خام. فعمودان — `contract_months` و`tenure_bucket` — صُنعا في اليوم الرابع **داخل دفتر**، قبل
أن يرى المسارُ البيانات. فالمُخرَج ناقص، ولا شيء فيه يقول ذلك. وتلك الفجوة هي بعينها شكل أول عطل في
كل منظومة تقديم حقيقية، والمهمة الأولى تجعلك تنظر إليها مباشرةً.

**وللنصف الثاني تحتاج بيانات مشروعك ونموذجك**، على أي حال كانا. أحضر المستودع لا لقطة شاشة.

</div>

## Setup

Everything today writes into `serving_demo/`, next to this notebook. By the end it holds four
scripts and a metadata file, and it is the artefact this lab produces: three ways to fail, one way
that works, and the evidence for each.

<div dir="rtl" align="right">

## الإعداد

كل شيء اليوم يُكتب في `serving_demo/` بجوار هذا الدفتر. وفي النهاية يحوي أربعة سكربتات وملف بيانات
وصفية، وهو المُخرَج الذي ينتجه هذا المعمل: ثلاث طرق للإخفاق، وطريقة واحدة تعمل، ودليل كلٍّ منها.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, versions
from aiep.data import load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, report

ensure("scikit-learn", "pandas", "pyarrow", "joblib")
seed_everything(42)

import json
import subprocess
import sys
import textwrap
from datetime import date
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DEMO = Path.cwd() / "serving_demo"
DEMO.mkdir(exist_ok=True)

FEATURES = pd.read_parquet(load_artefact("features.parquet"))
PIPELINE_PATH = Path(load_artefact("pipeline.joblib"))
PIPELINE = joblib.load(PIPELINE_PATH)

TARGET = "churn_flag"
NUMERIC = ["tenure", "MonthlyCharges", "TotalCharges", "contract_months", "SeniorCitizen"]
CATEGORICAL = ["Contract", "InternetService", "PaymentMethod", "tenure_bucket",
               "OnlineSecurity", "TechSupport", "PaperlessBilling", "gender", "Partner",
               "Dependents"]
COLUMNS = NUMERIC + CATEGORICAL

X = FEATURES[COLUMNS]
y = FEATURES[TARGET]

print(f"{len(FEATURES):,} rows · {len(COLUMNS)} model columns · target '{TARGET}'")
print(f"pipeline from {PIPELINE_PATH}")
print(f"steps: {list(PIPELINE.named_steps)}")
print(f"serving demo will be written to {DEMO}")
print(versions())

## Section 1 — Warm-up: the artifact you have, and what it is missing  (≈20 min)

Load the pipeline, predict on five customers, and print the probabilities. It works. Nothing in this
section is broken, and that is the point of it: **everything below fails from exactly this starting
position.**

Then ask the artifact what it needs. `pipeline.named_steps["preprocess"]` names the columns it
expects, and two of them — `contract_months` and `tenure_bucket` — do not exist in the raw
`telco_churn` file. They were built in a notebook in W2D4. So this artifact cannot predict from raw
data, and the file gives you no hint of that: it will simply raise a `KeyError` about a column, in
production, on a Sunday.

Write the missing feature function into `serving_demo/features.py` now, before anything else. It is
the smallest piece of today and the one most often skipped.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: المُخرَج الذي عندك، وما ينقصه (نحو ٢٠ دقيقة)

حمّل المسار، وتنبّأ لخمسة عملاء، واطبع الاحتمالات. إنه يعمل. ولا شيء في هذا القسم معطوب، وهذا
مقصوده: **فكل ما دونه يُخفق انطلاقًا من هذا الموضع بالضبط.**

ثم اسأل المُخرَج عمّا يحتاجه. فـ`pipeline.named_steps["preprocess"]` يسمّي الأعمدة التي يتوقّعها،
واثنان منها — `contract_months` و`tenure_bucket` — لا وجود لهما في ملف `telco_churn` الخام. فقد
صُنعا في دفتر في اليوم الرابع. فهذا المُخرَج لا يستطيع التنبّؤ من بيانات خام، والملف لا يلمّح إلى ذلك
البتّة: سيرفع ببساطة `KeyError` عن عمود، في الإنتاج، يوم أحد.

اكتب دالة الخصائص الناقصة في `serving_demo/features.py` الآن قبل كل شيء. فهي أصغر قطعة اليوم
وأكثر ما يُتخطّى.

</div>

In [ ]:
sample = X.head(5)
probabilities = PIPELINE.predict_proba(sample)[:, 1]
print("five customers, probability of churn:")
for i, probability in enumerate(probabilities):
    print(f"  row {i}: {probability:.4f}  (actual: {y.iloc[i]})")

expected_columns = []
for name, _, columns in PIPELINE.named_steps["preprocess"].transformers_:
    if isinstance(columns, list):
        expected_columns += columns
print(f"\nthe pipeline expects {len(expected_columns)} columns")

RAW_ONLY = {"contract_months", "tenure_bucket"}
print(f"of those, built outside the artifact in W2D4: {sorted(RAW_ONLY)}")
print("Nothing in pipeline.joblib records that. This is failure zero, and it ships more often\n"
      "than the three the session covered.")

# The module text below is data: it is written to disk and imported by another process,
# so it must not depend on anything this notebook has in memory.
FEATURES_PY = '''"""The feature engineering W2D4 did in a notebook, written down as code.

The pipeline expects contract_months and tenure_bucket. Neither is in the raw file, so
without this module the artifact cannot predict from anything a real service receives.
"""
import pandas as pd

CONTRACT_MONTHS = {"Month-to-month": 1, "One year": 12, "Two year": 24}
TENURE_EDGES = [-1, 12, 24, 48, 1000]
TENURE_LABELS = ["0-12", "13-24", "25-48", "49+"]


def add_features(frame):
    """Add the two engineered columns to a raw telco frame. Never mutates the input."""
    out = frame.copy()
    out["contract_months"] = out["Contract"].map(CONTRACT_MONTHS)
    out["tenure_bucket"] = pd.cut(out["tenure"], bins=TENURE_EDGES, labels=TENURE_LABELS)
    out["tenure_bucket"] = out["tenure_bucket"].astype(str)
    return out
'''
(DEMO / "features.py").write_text(FEATURES_PY, encoding="utf-8")
print(f"\nwrote {DEMO / 'features.py'} — {len(FEATURES_PY.splitlines())} lines that were "
      f"previously only in a notebook")

## Section 2 — Core: six tasks  (≈60 min)

1. The reference prediction — the number every later run has to match, exactly.
2. **Failure 1: weights only.** Save the estimator alone, load it in a fresh process, watch it raise.
3. **Failure 2: refit at inference.** No exception, no warning, and every prediction identical.
4. **The fix.** Save the whole pipeline, predict in a fresh process, compare to the reference.
5. `artifact_meta.json` — version, date, validation score, commit.
6. The unknown category: raising, ignoring, and which one you would ship.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. التنبّؤ المرجعي — الرقم الذي يجب أن يطابقه كل تشغيل لاحق تمامًا.
٢. **الإخفاق الأول: الأوزان وحدها.** احفظ المقدّر وحده، وحمّله في عملية جديدة، وشاهده يرفع خطأً.
٣. **الإخفاق الثاني: إعادة الملاءمة وقت الاستدلال.** لا استثناء ولا تحذير، وكل تنبّؤ مطابق للآخر.
٤. **الإصلاح.** احفظ المسار كاملًا، وتنبّأ في عملية جديدة، وقارن بالمرجع.
٥. `artifact_meta.json` — الإصدار والتاريخ ودرجة التحقّق والالتزام.
٦. الفئة المجهولة: الرفع، والتجاهل، وأيّهما تُسلِّم.

</div>

### Task 2.1 — the reference prediction

Pick one customer, keep the row, and record the pipeline's probability for it to full precision.
Every process below has to reproduce **this number**, and "close enough" is not a passing result:
a serving path that gives 0.7314 where training gave 0.7311 has a bug you have not found yet.

Write the row to `serving_demo/one_customer.json` so the scripts can read it without importing
anything from this notebook.

<div dir="rtl" align="right">

### المهمة ٢٫١ — التنبّؤ المرجعي

اختر عميلًا واحدًا، واحتفظ بصفّه، وسجّل احتمال المسار له بدقّته الكاملة. وعلى كل عملية أدناه أن
تُعيد **هذا الرقم**، و«قريب بما يكفي» ليست نتيجة ناجحة: فمسار تقديم يعطي 0.7314 حيث أعطى التدريب
0.7311 فيه خلل لم تجده بعد.

اكتب الصفّ في `serving_demo/one_customer.json` كي تقرأه السكربتات بلا استيراد شيء من هذا الدفتر.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Take one row as a one-row DataFrame — X.iloc[[i]] and not X.iloc[i], or the shape
#    is wrong and sklearn will tell you so at the least convenient moment.
# 2) predict_proba returns a column per class; the probability of churn is column 1.
# 3) Write the row to JSON with .to_dict(orient="records")[0]; json.dump needs plain
#    Python types, so cast numpy numbers with float() or int() first.
# Search: "pandas dataframe single row to_dict records json"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_dict.html
#
# ١) خذ صفًّا واحدًا كإطار من صفّ واحد — `X.iloc[[i]]` لا `X.iloc[i]`، وإلا كان الشكل خطأً
#    وأخبرك sklearn بذلك في أسوأ لحظة.
# ٢) تُعيد `predict_proba` عمودًا لكل فئة، واحتمال الانصراف هو العمود الأول (رقم ١).
# ٣) اكتب الصفّ إلى JSON بـ`.to_dict(orient="records")[0]`؛ ويحتاج `json.dump` أنواع
#    بايثون العادية، فحوّل أعداد numpy بـ`float()` أو `int()` أولًا.
# ابحث عن: "pandas dataframe single row to_dict records json"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_dict.html
# ────────────────────────────────────────────────────────────────────

ROW_INDEX = 7
# TODO: Take row ROW_INDEX as a one-row frame, record the pipeline's churn probability, and write the row to serving_demo/one_customer.json.
# مهمة: خذ الصفّ `ROW_INDEX` كإطار من صفّ واحد، وسجّل احتمال الانصراف من المسار، واكتب الصفّ إلى `serving_demo/one_customer.json`.
print(f"reference prediction for row {ROW_INDEX}: {REFERENCE!r}")
print(f"rounded, the way a dashboard would show it: {REFERENCE:.4f}")
print(f"\nwrote {DEMO / 'one_customer.json'}")
print(json.dumps(payload, indent=2)[:300], "…")

### Task 2.2 — failure 1: weights only, in a fresh process

Save **just the final estimator** — `PIPELINE.named_steps["model"]`, the fitted logistic regression
— to `serving_demo/model_only.joblib`. That is what people save when they think of the model as
"the thing that learned".

Then write `serving_demo/predict_weights_only.py`: it loads that file, reads `one_customer.json`,
and calls `predict_proba` on the raw row. Run it with `subprocess`, and print whatever it says.

**Why a subprocess and not another cell:** in this kernel, `PIPELINE` is still in memory, and so is
every fitted encoder inside it. A second cell that appears to work is testing your kernel's memory,
not your artifact. A fresh interpreter has your file and nothing else, which is exactly the
situation a service is in.

Expect a `ValueError` about converting a string to a float. Read it as the sentence it is: *the
one-hot encoder that used to turn `"Month-to-month"` into numbers is not in this file.*

<div dir="rtl" align="right">

### المهمة ٢٫٢ — الإخفاق الأول: الأوزان وحدها في عملية جديدة

احفظ **المقدّر النهائي وحده** — `PIPELINE.named_steps["model"]`، أي الانحدار اللوجستي المُدرَّب — في
`serving_demo/model_only.joblib`. وهذا ما يحفظه الناس حين يتصوّرون النموذج «الشيءَ الذي تعلّم».

ثم اكتب `serving_demo/predict_weights_only.py`: يحمّل ذلك الملف، ويقرأ `one_customer.json`، وينادي
`predict_proba` على الصفّ الخام. وشغّله بـ`subprocess`، واطبع ما يقوله.

**ولماذا عملية منفصلة لا خلية أخرى:** لأن `PIPELINE` ما يزال في ذاكرة هذه النواة، ومعه كل مُرمِّز
مُدرَّب داخله. فالخلية الأخرى التي تبدو ناجحة تختبر ذاكرة نواتك لا مُخرَجك. أما المفسّر الجديد فليس
عنده إلا ملفك، وهذا بالضبط موضع الخدمة.

وتوقّع `ValueError` عن تحويل نصّ إلى عدد عشري. واقرأه على أنه الجملة التي هو: *المُرمِّز الأحادي الذي
كان يحوّل `"Month-to-month"` إلى أعداد ليس في هذا الملف.*

</div>

In [ ]:
# The script below is data, not exercise code: it is written to disk and run by a separate
# interpreter, so it may not use anything this notebook has in memory.
WEIGHTS_ONLY_SCRIPT = '''import json
from pathlib import Path

import joblib
import pandas as pd

here = Path(__file__).parent
model = joblib.load(here / "model_only.joblib")
row = pd.DataFrame([json.loads((here / "one_customer.json").read_text())])
print("loaded a", type(model).__name__, "and one row of", row.shape[1], "columns")
print("probability:", model.predict_proba(row)[0, 1])
'''
print(WEIGHTS_ONLY_SCRIPT)

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) joblib.dump(PIPELINE.named_steps["model"], DEMO / "model_only.joblib") — the
#    estimator only, no preprocessing.
# 2) Write the script as a plain string and .write_text() it. It must import nothing
#    from this notebook: json, joblib and pandas, and the two files on disk.
# 3) subprocess.run([sys.executable, path], capture_output=True, text=True) gives you
#    .returncode, .stdout and .stderr. Print the last few lines of stderr.
# Search: "python subprocess run capture_output returncode"
# https://docs.python.org/3/library/subprocess.html#subprocess.run
#
# ١) `joblib.dump(PIPELINE.named_steps["model"], DEMO / "model_only.joblib")` — المقدّر
#    وحده بلا معالجة مسبقة.
# ٢) اكتب السكربت نصًّا عاديًّا واحفظه بـ`.write_text()`. ويجب ألّا يستورد شيئًا من هذا
#    الدفتر: `json` و`joblib` و`pandas`، والملفّان على القرص.
# ٣) `subprocess.run([sys.executable, path], capture_output=True, text=True)` يعطيك
#    `.returncode` و`.stdout` و`.stderr`. اطبع آخر أسطر `stderr`.
# ابحث عن: "python subprocess run capture_output returncode"
# https://docs.python.org/3/library/subprocess.html#subprocess.run
# ────────────────────────────────────────────────────────────────────

# TODO: Save the estimator alone, write the script, and run it in a fresh interpreter.
# مهمة: احفظ المقدّر وحده، واكتب السكربت، وشغّله في مفسّر جديد.
print(f"exit code: {WEIGHTS_ONLY.returncode}")
print(WEIGHTS_ONLY.stdout)
print("stderr, last lines:")
print(textwrap.indent("\n".join(WEIGHTS_ONLY.stderr.strip().splitlines()[-4:]), "  "))
WEIGHTS_ONLY_FAILED = WEIGHTS_ONLY.returncode != 0
print(f"\nfailed in a fresh process: {WEIGHTS_ONLY_FAILED} — and the traceback names a string "
      f"it could not turn into a number, which is the encoder you did not save.")

### Task 2.3 — failure 2: refit at inference, which does not raise anything

This is the one that ships.

The service receives a row, notices the numbers need scaling, and does the obvious thing: fits a
`StandardScaler` on the incoming data and transforms it. No exception. A number comes out. Someone
puts it on a dashboard.

Fit a scaler **on the single incoming row** and transform it. Every scaled feature is `0.0` — a
single sample is its own mean. Then run twenty *different* customers through the same path, one at
a time, and print all twenty predictions.

**They are identical.** Twenty different people, one number, no error anywhere. Compare them against
the reference and against each other, and note that the only way to catch this is a test that asks
whether different inputs produce different outputs.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — الإخفاق الثاني: إعادة الملاءمة وقت الاستدلال، ولا ترفع شيئًا

وهذا هو الذي يصل إلى الإنتاج.

تستقبل الخدمة صفًّا، فتلاحظ أن الأعداد تحتاج تقييسًا، فتفعل الشيء البديهي: تلائم `StandardScaler`
على البيانات الواردة وتحوّلها. لا استثناء. ويخرج رقم. ويضعه أحدهم على لوحة.

لائِم مقيِّسًا **على الصفّ الوارد وحده** وحوّله. فتكون كل خاصّية مقيَّسة `0.0` — إذ العيّنة الواحدة
متوسّط نفسها. ثم مرّر عشرين عميلًا *مختلفين* في المسار نفسه، واحدًا واحدًا، واطبع التنبّؤات العشرين.

**إنها متطابقة.** عشرون شخصًا مختلفًا، ورقم واحد، ولا خطأ في أي موضع. قارنها بالمرجع وبعضها ببعض،
ولاحظ أن السبيل الوحيد لاكتشاف هذا اختبارٌ يسأل هل تُنتج المُدخَلات المختلفة مخرجاتٍ مختلفة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Train a small numeric-only logistic regression here so the demo is self-contained:
#    fit a StandardScaler on X[NUMERIC] and a LogisticRegression on the scaled values.
# 2) The failure is refitting: for each incoming row, call StandardScaler().fit_transform
#    on that row alone and predict from the result.
# 3) Loop over 20 rows, collect the probabilities, and check how many distinct values
#    came out. np.unique(np.round(values, 12)) is enough.
# Search: "standardscaler fit on single sample zeros"
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
#
# ١) درّب انحدارًا لوجستيًّا صغيرًا على الأعمدة العددية وحدها ليكون العرض مكتفيًا بذاته:
#    لائِم `StandardScaler` على `X[NUMERIC]` و`LogisticRegression` على القيم المقيَّسة.
# ٢) والإخفاق هو إعادة الملاءمة: لكل صفّ وارد نادِ `StandardScaler().fit_transform` على
#    ذلك الصفّ وحده وتنبّأ من الناتج.
# ٣) ودُر على عشرين صفًّا، واجمع الاحتمالات، وتحقّق كم قيمة مميّزة خرجت. و
#    `np.unique(np.round(values, 12))` كافٍ.
# ابحث عن: "standardscaler fit on single sample zeros"
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
# ────────────────────────────────────────────────────────────────────

numeric_frame = X[NUMERIC].fillna(X[NUMERIC].median())
fitted_scaler = StandardScaler().fit(numeric_frame)
numeric_model = LogisticRegression(max_iter=2000).fit(
    fitted_scaler.transform(numeric_frame), y)
# TODO: Predict 20 different customers the wrong way — refitting a scaler on each single incoming row — and collect the probabilities.
# مهمة: تنبّأ لعشرين عميلًا مختلفًا بالطريقة الخطأ — بإعادة ملاءمة مقيِّس على كل صفّ وارد وحده — واجمع الاحتمالات.
print(f"the single incoming row after refitting the scaler: {first_scaled.round(6)}")
print("every feature is exactly zero — one sample is its own mean\n")
print("twenty different customers, refit-per-request:")
print("  " + ", ".join(f"{p:.6f}" for p in REFIT_PREDICTIONS[:8]) + " …")
print(f"  distinct predictions among all 20: {DISTINCT}")
correct = numeric_model.predict_proba(fitted_scaler.transform(numeric_frame.iloc[:20]))[:, 1]
print(f"\nthe same twenty with the FITTED scaler: "
      f"{', '.join(f'{p:.4f}' for p in correct[:8])} …")
print(f"  distinct: {len(np.unique(np.round(correct, 12)))}")
print("\nNo exception was raised on either path. The broken one is the one that looks calm.")

### Task 2.4 — the fix, verified where it counts

Save the **whole fitted pipeline** — preprocessing and estimator together — write
`serving_demo/predict.py` that loads it, reads `one_customer.json`, applies `features.py` if the row
needs it, and prints the probability. Run it in a fresh process and compare to the reference from
task 1.

The comparison is `==`, not `np.isclose`. The same objects on the same input must give the same
float. If they do not, something in your serving path is not the thing you trained, and finding out
which part is a much better use of an afternoon than shipping.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — الإصلاح، مُتحقَّقًا منه حيث يهمّ

احفظ **المسار المُدرَّب كاملًا** — المعالجة المسبقة والمقدّر معًا — واكتب `serving_demo/predict.py`
يحمّله، ويقرأ `one_customer.json`، ويطبّق `features.py` إن احتاج الصفّ ذلك، ويطبع الاحتمال. ثم
شغّله في عملية جديدة وقارن بالمرجع من المهمة الأولى.

والمقارنة بـ`==` لا بـ`np.isclose`. فالكائنات نفسها على المُدخَل نفسه يجب أن تعطي العدد نفسه. وإن لم
تفعل فشيء في مسار تقديمك ليس ما درّبته، ومعرفة أي جزء هو أفضل استعمالًا لظهيرة من التسليم.

</div>

In [ ]:
# Again: this text becomes serving_demo/predict.py, and another interpreter runs it.
PREDICT_SCRIPT = '''"""Predict churn for one customer. Run: python serving_demo/predict.py"""
import json
from pathlib import Path

import joblib
import pandas as pd

HERE = Path(__file__).parent


def load():
    """One loading function. The interface, the batch job and the tests all call this."""
    return joblib.load(HERE / "pipeline.joblib")


def predict(record, pipeline=None):
    """Probability of churn for one record given as a dict."""
    pipeline = pipeline or load()
    frame = pd.DataFrame([record])
    if "contract_months" not in frame.columns:
        from features import add_features
        frame = add_features(frame)
    return float(pipeline.predict_proba(frame)[0, 1])


if __name__ == "__main__":
    record = json.loads((HERE / "one_customer.json").read_text())
    print(repr(predict(record)))
'''
print(PREDICT_SCRIPT[:220], "…")

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) joblib.dump(PIPELINE, DEMO / "pipeline.joblib") — the whole object, not a step of it.
# 2) The script prints the probability with repr() so no digits are lost. A dashboard
#    rounds; a verification script must not.
# 3) Compare the printed float to REFERENCE with == and print both. If they differ, print
#    the difference — you will want to see the exponent.
# Search: "joblib dump load full sklearn pipeline"
# https://scikit-learn.org/stable/model_persistence.html
#
# ١) `joblib.dump(PIPELINE, DEMO / "pipeline.joblib")` — الكائن كله لا خطوةً منه.
# ٢) ويطبع السكربت الاحتمال بـ`repr()` كي لا تضيع خانات. فاللوحة تقرّب، وسكربت التحقّق
#    لا يجوز له.
# ٣) وقارن العدد المطبوع بـ`REFERENCE` بـ`==` واطبع الاثنين. وإن اختلفا فاطبع الفرق —
#    فستريد أن ترى الأُسّ.
# ابحث عن: "joblib dump load full sklearn pipeline"
# https://scikit-learn.org/stable/model_persistence.html
# ────────────────────────────────────────────────────────────────────

# TODO: Save the whole pipeline beside the script, run predict.py in a fresh process, and compare its output to REFERENCE exactly.
# مهمة: احفظ المسار كاملًا بجوار السكربت، وشغّل `predict.py` في عملية جديدة، وقارن مخرجه بـ`REFERENCE` تمامًا.
print(f"exit code {FRESH.returncode}")
print(f"  fresh process: {FRESH_VALUE!r}")
print(f"  notebook:      {REFERENCE!r}")
print(f"  identical:     {EXACT_MATCH}")
if not EXACT_MATCH:
    print(f"  difference:    {FRESH_VALUE - REFERENCE:.3e}")
print(f"\n{FRESH.stderr.strip()[-200:] if FRESH.stderr.strip() else 'no stderr'}")

### Task 2.5 — `artifact_meta.json`, four fields

An artifact with no metadata is a file whose predictions you cannot explain in three months. Write
four fields beside it:

| Field | Why it is there |
|---|---|
| `version` | So two artifacts can be told apart when both are on the server |
| `trained_on` | The date the data ends, not the date you ran it — those differ, and only one matters |
| `validation_score` | The number you claimed. If it is not next to the file, it will be misremembered upwards |
| `git_commit` | The code that produced it. Without this the other three are decoration |

Get the commit from `git rev-parse HEAD` through `subprocess`, and handle the case where there is no
repository — that path runs on a student machine, so it has to say something useful rather than
raise.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — `artifact_meta.json` بأربعة حقول

المُخرَج بلا بيانات وصفية ملفٌّ لا تستطيع تفسير تنبّؤاته بعد ثلاثة أشهر. فاكتب أربعة حقول بجواره:
الإصدار، وتاريخ انتهاء البيانات، ودرجة التحقّق، والتزام git.

خذ الالتزام من `git rev-parse HEAD` عبر `subprocess`، وعالج حالة عدم وجود مستودع — فهذا المسار يعمل
على جهاز طالب، فعليه أن يقول شيئًا مفيدًا لا أن يرفع خطأً.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True) — check
#    returncode before trusting stdout, and fall back to "unknown (not a git checkout)".
# 2) The validation score should be a real number you measured, not a remembered one:
#    score the loaded pipeline on a held-out slice here and use that.
# 3) json.dump with indent=2 so a human can read it in a code review.
# Search: "python subprocess git rev-parse HEAD"
# https://docs.python.org/3/library/subprocess.html
#
# ١) `subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True)` —
#    تحقّق من `returncode` قبل الوثوق بـ`stdout`، وارجع إلى "unknown (not a git checkout)".
# ٢) ويجب أن تكون درجة التحقّق رقمًا قِسته فعلًا لا رقمًا تذكرته: فقيّم المسار المُحمَّل على
#    شريحة محجوزة هنا واستعمل الناتج.
# ٣) و`json.dump` بـ`indent=2` كي يقرأه إنسان في مراجعة رمز.
# ابحث عن: "python subprocess git rev-parse HEAD"
# https://docs.python.org/3/library/subprocess.html
# ────────────────────────────────────────────────────────────────────

# TODO: Build artifact_meta.json with version, trained_on, validation_score and git_commit, measuring the score rather than remembering it.
# مهمة: ابنِ `artifact_meta.json` بالإصدار وتاريخ البيانات ودرجة التحقّق والتزام git، بقياس الدرجة لا بتذكّرها.
RELOADED = json.loads((DEMO / "artifact_meta.json").read_text())
for field in ["version", "trained_on", "validation_score", "git_commit"]:
    print(f"  {field:<18} {RELOADED[field]}")
print(f"\nall four present: "
      f"{all(RELOADED.get(f) for f in ['version', 'trained_on', 'validation_score', 'git_commit'])}")

### Task 2.6 — the category nobody trained on

A new contract type appears — `"Weekly"` — because the business launched it on Tuesday and told
nobody.

Show both behaviours. Build a `OneHotEncoder(handle_unknown="error")` on the training categories and
push the unknown value through it: it raises, loudly, at the door. Then use the shipped pipeline,
whose encoder was built with `handle_unknown="ignore"`: it returns a prediction, computed from a row
where the contract column contributed **nothing at all**.

Then write two sentences on which one a real service should do. "Never crash" is not automatically
the right answer: a prediction made from a silently zeroed feature is a wrong answer delivered with
the same confidence as a right one, and the caller cannot tell them apart. The defensible designs
are *fail loudly* or *answer, and say the input was out of distribution* — and the second one needs
you to have built the flag.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — الفئة التي لم يتدرّب عليها أحد

يظهر نوع عقد جديد — `"Weekly"` — لأن العمل أطلقه الثلاثاء ولم يخبر أحدًا.

أظهِر السلوكين. ابنِ `OneHotEncoder(handle_unknown="error")` على فئات التدريب ومرّر القيمة المجهولة
فيه: فيرفع خطأً عاليًا عند الباب. ثم استعمل المسار المُسلَّم الذي بُني مُرمِّزه بـ
`handle_unknown="ignore"`: فيعيد تنبّؤًا محسوبًا من صفّ لم يُسهم فيه عمود العقد **بشيء البتّة**.

ثم اكتب جملتين عن أيّهما ينبغي للخدمة الحقيقية. و«لا تنهار أبدًا» ليس الجواب الصحيح تلقائيًّا: فالتنبّؤ
المحسوب من خاصّية صُفِّرت بصمت جوابٌ خاطئ يُسلَّم بثقة الجواب الصحيح نفسها، ولا يستطيع المستدعي أن
يفرّق. والتصميمان اللذان يُدافَع عنهما هما *الإخفاق العالي* أو *الإجابة مع الإفصاح أن المُدخَل خارج
التوزيع* — والثاني يحتاج أن تكون قد بنيت العلامة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Copy the reference row, set Contract to "Weekly", and keep everything else as it was.
# 2) OneHotEncoder(handle_unknown="error").fit(X[["Contract"]]) then .transform on the
#    unknown value — wrap it in try/except ValueError and print the message.
# 3) For the shipped pipeline, predict and then compare against the same row with its
#    original contract. The difference tells you how much the zeroed column was worth.
# Search: "OneHotEncoder handle_unknown ignore error"
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html
#
# ١) انسخ الصفّ المرجعي، واجعل `Contract` مساويًا `"Weekly"`، وأبقِ الباقي كما كان.
# ٢) `OneHotEncoder(handle_unknown="error").fit(X[["Contract"]])` ثم `.transform` على
#    القيمة المجهولة — ضعه في `try/except ValueError` واطبع الرسالة.
# ٣) وللمسار المُسلَّم، تنبّأ ثم قارن بالصفّ نفسه بعقده الأصلي. والفرق يخبرك كم كانت قيمة
#    العمود المصفَّر.
# ابحث عن: "OneHotEncoder handle_unknown ignore error"
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html
# ────────────────────────────────────────────────────────────────────

UNKNOWN_ROW = ONE_ROW.copy()
UNKNOWN_ROW["Contract"] = "Weekly"
# TODO: Show the strict encoder raising on the unknown category, then the shipped pipeline predicting from it anyway, and record both outcomes.
# مهمة: أظهِر المُرمِّز الصارم يرفع خطأً على الفئة المجهولة، ثم المسار المُسلَّم يتنبّأ منها رغم ذلك، وسجّل النتيجتين.
print(f"handle_unknown='error'  → raised: {STRICT_RAISED}")
print(f"    {STRICT_MESSAGE[:110]}")
print(f"\nhandle_unknown='ignore' → {UNKNOWN_PREDICTION:.4f} "
      f"(the same customer with a known contract: {REFERENCE:.4f})")
print(f"    the contract column contributed nothing, and the answer moved by "
      f"{abs(UNKNOWN_PREDICTION - REFERENCE):.4f}")
print("\nNothing in the second output says a feature was dropped. That is the design question.")

**Your two sentences.** Which should a service do with an unseen category, and why is "never crash"
not automatically right? Replace this text.

> …

<div dir="rtl" align="right">

**جملتاك.** ماذا ينبغي للخدمة أن تفعل بفئة لم ترَها، ولماذا ليست «لا تنهار أبدًا» صحيحةً تلقائيًّا؟
استبدل هذا النصّ.

> …

</div>

## Section 3 — Your capstone  (≈90 min, and this is the half that is graded)

Nothing below is checked by an assertion. It is checked by a TA walking the room, and by demo day.
Work on **your own project** and tick these off in order — each one is a separate failure mode that
has ended a demo:

- [ ] Your capstone artifact saves **weights and every fitted transform**, in one file or one
      directory, loaded by one function.
- [ ] A `predict.py` loads it and predicts on one new record **from the command line**.
- [ ] It runs in a fresh terminal, in a fresh clone, with no notebook open and no notebook imported.
- [ ] `artifact_meta.json` exists beside the artifact with the four fields from task 5.
- [ ] One interface — Streamlit **or** FastAPI, not both — and it calls the same `load()` and
      `predict()` your command line does.
- [ ] Your README's first code block is the thing a stranger runs first, and you have watched it
      work in a fresh clone.
- [ ] `git status` is clean: no data, no artifacts over 50 MB, no `.env`, no keys.

**Stretch, tonight (~30 min).** Add input validation and an out-of-distribution response: when the
model's confidence is below a threshold you choose, return `"uncertain"` rather than a label. This
is the third time this idea has arrived — W4D5's three out-of-class images, W7D3's refusal
instruction, and now yours. A system that can say "I don't know" is a different class of system from
one that cannot, and demo day notices.

<div dir="rtl" align="right">

## القسم الثالث — مشروعك (نحو ٩٠ دقيقة، وهذا هو النصف المُقيَّم)

لا شيء أدناه يفحصه تأكيد. يفحصه مساعدٌ يجول في القاعة، ويفحصه يوم العرض. اعمل على **مشروعك أنت**
وأنجز هذه بالترتيب — فكلٌّ منها صنف إخفاق أنهى عرضًا من قبل:

- [ ] يحفظ مُخرَج مشروعك **الأوزان وكل تحويل مُدرَّب**، في ملف واحد أو مجلّد واحد، تحمّله دالّة واحدة.
- [ ] وملف `predict.py` يحمّله ويتنبّأ لسجلّ جديد **من سطر الأوامر**.
- [ ] ويعمل في طرفية جديدة، وفي نسخة مستودع جديدة، بلا دفتر مفتوح ولا دفتر مستورد.
- [ ] و`artifact_meta.json` موجود بجوار المُخرَج بالحقول الأربعة من المهمة الخامسة.
- [ ] وواجهة واحدة — Streamlit **أو** FastAPI لا كلاهما — تنادي `load()` و`predict()` نفسيهما اللذين
      يناديهما سطر الأوامر.
- [ ] وأول كتلة رمز في ملف README هي ما يشغّله الغريب أولًا، وقد رأيتها تعمل في نسخة جديدة.
- [ ] و`git status` نظيف: لا بيانات، ولا مُخرَجات فوق ٥٠ ميغابايت، ولا `.env`، ولا مفاتيح.

**التوسّع الليلة (نحو ٣٠ دقيقة).** أضف تحقّقًا من المُدخَل واستجابةً لخارج التوزيع: فإذا كانت ثقة
النموذج دون عتبة تختارها، أعِد `"uncertain"` بدل تسمية. وهذه ثالث مرّة تصل فيها هذه الفكرة — صور
الأسبوع الرابع الثلاث خارج الفئات، وتعليمة الرفض في الأسبوع السابع، والآن مشروعك. والنظام الذي
يستطيع أن يقول «لا أعرف» صنفٌ مختلف عن الذي لا يستطيع، ويوم العرض يلاحظ ذلك.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Wrap the prediction: get the probability, and if it sits between the two thresholds
#    you choose, return "uncertain" instead of a label.
# 2) Pick the thresholds from data, not from taste: look at where the model's probabilities
#    actually fall and what accuracy looks like inside the band you are about to refuse.
# 3) Report the cost: how many rows would now be refused, and what the accuracy is on the
#    ones you still answer. A refusal rate with no accuracy beside it says nothing.
# Search: "prediction confidence threshold abstain selective classification"
# https://scikit-learn.org/stable/modules/model_evaluation.html
#
# ١) غلّف التنبّؤ: خذ الاحتمال، وإن وقع بين العتبتين اللتين تختارهما فأعِد `"uncertain"`
#    بدل تسمية.
# ٢) واختر العتبتين من البيانات لا من الذوق: انظر أين تقع احتمالات النموذج فعلًا، وكيف
#    تبدو الدقّة داخل النطاق الذي توشك أن ترفضه.
# ٣) واذكر الثمن: كم صفًّا سيُرفض الآن، وكم الدقّة على ما تزال تجيب عنه. فمعدّل رفض بلا
#    دقّة بجواره لا يقول شيئًا.
# ابحث عن: "prediction confidence threshold abstain selective classification"
# https://scikit-learn.org/stable/modules/model_evaluation.html
# ────────────────────────────────────────────────────────────────────

# TODO: Add an abstain band around 0.5, then report the refusal rate and the accuracy on the rows you still answer, against the accuracy on everything.
# مهمة: أضف نطاق امتناع حول ٠٫٥، ثم اذكر معدّل الرفض والدقّة على الصفوف التي ما تزال تجيب عنها، مقابل الدقّة على الكل.
print(f"abstain band [{LOW}, {HIGH}]")
print(f"  refused:            {REFUSAL_RATE:.1%} of rows")
print(f"  accuracy overall:   {ALL_ACCURACY:.4f}")
print(f"  accuracy answered:  {ANSWERED_ACCURACY:.4f} "
      f"({ANSWERED_ACCURACY - ALL_ACCURACY:+.4f})")
print("\nThat trade is the whole design: you bought accuracy on the answers with coverage.\n"
      "Whether it is a good trade depends on what happens to a refused row, which is a\n"
      "product question and belongs in your report, not in the model.")

## Save the artefact

`serving_demo/` is the artefact: two scripts that fail in the two different ways, one that works,
the pipeline, the metadata, and the customer they all predict on. Keep it — it is the shortest
possible answer to "why does my model behave differently in the app", and you will need that answer
again.

<div dir="rtl" align="right">

## احفظ المُخرَج

`serving_demo/` هو المُخرَج: سكربتان يُخفقان بالطريقتين المختلفتين، وواحد يعمل، والمسار، والبيانات
الوصفية، والعميل الذي يتنبّأ له الجميع. احتفظ به — فهو أقصر جواب ممكن عن «لماذا يسلك نموذجي سلوكًا
مختلفًا في التطبيق»، وستحتاج ذلك الجواب ثانيةً.

</div>

In [ ]:
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY = pd.DataFrame([
    {"path": "weights only, fresh process", "raised": WEIGHTS_ONLY_FAILED,
     "prediction": None, "matches_reference": False},
    {"path": "refit scaler per request", "raised": False,
     "prediction": REFIT_PREDICTIONS[0], "matches_reference": False},
    {"path": "full pipeline, fresh process", "raised": False,
     "prediction": FRESH_VALUE, "matches_reference": EXACT_MATCH},
])
SUMMARY.to_parquet(ARTEFACT_DIR / "serving_results.parquet", index=False)

print(SUMMARY.to_string(index=False))
print(f"\nserving_demo/ contains:")
for path in sorted(DEMO.iterdir()):
    print(f"  {path.name:<28} {path.stat().st_size:>8,} bytes")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(WEIGHTS_ONLY_FAILED and "ValueError" in WEIGHTS_ONLY.stderr,
      f"the weights-only artifact must fail in a fresh process — exit code "
      f"{WEIGHTS_ONLY.returncode}, ValueError in stderr: "
      f"{'ValueError' in WEIGHTS_ONLY.stderr}. If it succeeded, the script is reading something "
      f"this notebook prepared, and the failure has not been reproduced",
      f"يجب أن يُخفق مُخرَج الأوزان وحدها في عملية جديدة — ورمز الخروج {WEIGHTS_ONLY.returncode}، "
      f"ووجود ValueError في stderr: {'ValueError' in WEIGHTS_ONLY.stderr}. وإن نجح فالسكربت يقرأ "
      f"شيئًا هيّأه هذا الدفتر، ولم يُعَد إنتاج الإخفاق")

check(DISTINCT == 1,
      f"refitting the scaler on each incoming row must give all 20 customers the same prediction — "
      f"got {DISTINCT} distinct values. One value from twenty different people is the silent "
      f"failure this task exists to show; more than one means the scaler saw more than one row",
      f"يجب أن تعطي إعادة ملاءمة المقيِّس على كل صفّ وارد التنبّؤ نفسه للعشرين جميعًا — والناتج "
      f"{DISTINCT} قيمة مميّزة. فقيمةٌ واحدة من عشرين شخصًا مختلفًا هي الإخفاق الصامت الذي وُجدت له "
      f"هذه المهمة، وأكثر من واحدة تعني أن المقيِّس رأى أكثر من صفّ")

check(FRESH.returncode == 0 and EXACT_MATCH,
      f"the full pipeline must load in a fresh process and give exactly the notebook's number — "
      f"exit {FRESH.returncode}, {FRESH_VALUE!r} against {REFERENCE!r}. Exactly, not nearly: a "
      f"difference in the last decimals means the serving path is not the training path",
      f"يجب أن يُحمَّل المسار الكامل في عملية جديدة وأن يعطي رقم الدفتر تمامًا — والخروج "
      f"{FRESH.returncode}، و{FRESH_VALUE!r} مقابل {REFERENCE!r}. تمامًا لا تقريبًا: فاختلاف في "
      f"الخانات الأخيرة يعني أن مسار التقديم ليس مسار التدريب")

check(all(RELOADED.get(field) for field in
          ["version", "trained_on", "validation_score", "git_commit"])
      and RELOADED["git_commit"] != "",
      f"artifact_meta.json must carry all four fields with real values — got "
      f"{ {k: RELOADED.get(k) for k in ['version', 'trained_on', 'validation_score', 'git_commit']} }. "
      f"A commit of 'unknown (not a git checkout)' is an acceptable value and an unacceptable "
      f"situation for a capstone you are about to hand in",
      f"يجب أن يحمل `artifact_meta.json` الحقول الأربعة بقيم حقيقية. والقيمة "
      f"'unknown (not a git checkout)' مقبولة كقيمة وغير مقبولة كحال لمشروع توشك أن تسلّمه")

check(STRICT_RAISED and isinstance(UNKNOWN_PREDICTION, float),
      f"the unknown category must be recorded both ways — the strict encoder raised: "
      f"{STRICT_RAISED}, and the shipped pipeline returned {UNKNOWN_PREDICTION:.4f} without "
      f"mentioning that a feature was dropped. Both behaviours are defensible; only one of them "
      f"is a decision you made",
      f"يجب تسجيل الفئة المجهولة بالطريقتين — فالمُرمِّز الصارم رفع خطأً: {STRICT_RAISED}، وأعاد "
      f"المسار المُسلَّم {UNKNOWN_PREDICTION:.4f} بلا ذكر أن خاصّية أُسقطت. وكلا السلوكين يُدافَع عنه، "
      f"وواحدٌ منهما فقط قرارٌ اتخذته أنت")

check((DEMO / "predict.py").exists() and (DEMO / "features.py").exists()
      and (DEMO / "artifact_meta.json").exists() and (DEMO / "pipeline.joblib").exists(),
      f"serving_demo/ must contain the four files a stranger needs: predict.py, features.py, "
      f"pipeline.joblib and artifact_meta.json — found "
      f"{sorted(p.name for p in DEMO.iterdir())}",
      f"يجب أن يحوي `serving_demo/` الملفات الأربعة التي يحتاجها الغريب: `predict.py` و`features.py` "
      f"و`pipeline.joblib` و`artifact_meta.json` — والموجود "
      f"{sorted(p.name for p in DEMO.iterdir())}")

report()

## What's next

**Tomorrow: clinic.** Your project goes on the screen in front of the room, and the first thing the
instructor does is clone it and follow your README out loud without asking you anything. Whatever
breaks there is finding number one, and it is almost always something today's checklist would have
caught.

Then Wednesday's report workshop, Thursday's timed rehearsal and **feature freeze**, and Friday you
demo — ten minutes, three of them live, on data the model has not seen.

The three questions you will be asked are already known, so prepare them now: **what is your
baseline and what does it score**, **show me a case where it gets it wrong**, and **what would you
do with two more weeks**.

<div dir="rtl" align="right">

## ما التالي

**غدًا: العيادة.** يظهر مشروعك على الشاشة أمام القاعة، وأول ما يفعله المدرّب أن ينسخ المستودع
ويتبع ملف README بصوت مسموع بلا أن يسألك شيئًا. وما يتعطّل هناك هو الملاحظة الأولى، وهو في الغالب
شيءٌ كانت قائمة اليوم لتلتقطه.

ثم ورشة التقرير الأربعاء، والبروفة المؤقّتة **وتجميد المزايا** الخميس، والجمعة تعرض — عشر دقائق،
ثلاث منها حيّة، على بيانات لم يرها النموذج.

والأسئلة الثلاثة التي ستُسأل معروفة سلفًا، فجهّزها الآن: **ما خطّ أساسك وكم يسجّل**، و**أرني حالةً
يُخطئ فيها**، و**ماذا كنت ستفعل بأسبوعين إضافيين**.

</div>